# AuralGuard — Kaggle Training Pipeline
Trains B1 → B2 → B3 → B5 → AuralGuard sequentially on Kaggle's free P100 GPU.

**Estimated total: ~14 hours.**
- B1 (LFCC-LCNN): ~30 min
- B2 (RawNet2): ~1 hr
- B3 (AASIST): ~2 hrs
- B5 (WavLM+AASIST): ~4 hrs
- AuralGuard (full): ~6 hrs

**Tracker:** Check cell execution time after each step.
If a cell runs >2x its estimate, abort and debug.

In [ ]:
# ── Progress helpers ──
import time, sys, subprocess, shlex
from datetime import datetime, timedelta
from pathlib import Path

def ts():
    return datetime.now().strftime("[%H:%M:%S]")

def log(msg):
    print(f"{ts()} {msg}", flush=True)

def run_cmd(cmd, timeout_min=None, cwd=None):
    log(f"$ {cmd}")
    start = time.time()
    proc = subprocess.Popen(
        cmd if isinstance(cmd, list) else shlex.split(cmd),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        cwd=cwd, text=True, bufsize=1
    )
    last_output = time.time()
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
        if line.strip():
            last_output = time.time()
        if timeout_min and (time.time() - start) > timeout_min * 60:
            proc.kill()
            log(f"TIMEOUT after {timeout_min} min -- aborting")
            return False
    proc.wait()
    elapsed = time.time() - start
    ok = proc.returncode == 0
    status = "OK" if ok else f"FAILED (code {proc.returncode})"
    log(f"Done in {elapsed:.0f}s -- {status}")
    return ok

try:
    from tqdm.auto import tqdm
except ImportError:
    !pip install tqdm -q
    from tqdm.auto import tqdm

log("Helpers loaded")

## 1. Setup
Finds and symlinks the dataset (instant), then verifies structure.

In [ ]:
import os

# Efficient search: probe known paths + shallow rglob
log("Searching for ASVspoof 2019 dataset...")
INPUT_DIR = None

# 1. Try exact paths user reported
EXACT_PATHS = [
    Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA"),
    Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset"),
    Path("/kaggle/input/asvpoof-2019-dataset/LA"),
    Path("/kaggle/input/asvpoof-2019-dataset"),
]
for p in EXACT_PATHS:
    if (p / "ASVspoof2019_LA_cm_protocols").exists():
        INPUT_DIR = p
        log(f"  Found at {p} ✓")
        break

# 2. If not found, probe known wrapper structures
if INPUT_DIR is None:
    for root in [Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset"),
                 Path("/kaggle/input/asvpoof-2019-dataset")]:
        if not root.exists():
            continue
        log(f"  Probing under {root.name}...")
        for sub in ["", "LA", "asvpoof-2019-dataset", "asvpoof-2019-dataset/LA"]:
            candidate = root / sub if sub else root
            proto = candidate / "ASVspoof2019_LA_cm_protocols"
            if candidate.exists() and proto.exists():
                INPUT_DIR = candidate
                log(f"  Matched at {candidate} ✓")
                break
        if INPUT_DIR:
            break

if INPUT_DIR is None:    log("ERROR: Dataset not found!")    log("  Listing /kaggle/input contents:")    for p in Path("/kaggle/input").iterdir():        log(f"    {p.name}  dir={p.is_dir()}")    # List dataset root contents for debugging    for candidate_root in [Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset"),                           Path("/kaggle/input/asvpoof-2019-dataset")]:        if candidate_root.exists():            log(f"  Contents of {candidate_root}:")            for child in sorted(candidate_root.iterdir()):                log(f"    {child.name}  dir={child.is_dir()}  size={child.stat().st_size if child.is_file() else 0}")    # Recursive search for protocol directory    log("  Recursive search for '*cm_protocols*'...")    for hit in Path("/kaggle/input").rglob("*cm_protocols*"):        log(f"    FOUND: {hit}  dir={hit.is_dir()}")    log("  -> Click +Data in right sidebar, search 'awsaf49/asvpoof-2019-dataset', click Add")    raise FileNotFoundError(f"Dataset not found at any expected path")
# Show dataset structure
expected = ["ASVspoof2019_LA_train", "ASVspoof2019_LA_dev",
            "ASVspoof2019_LA_eval", "ASVspoof2019_LA_cm_protocols"]
for d in expected:
    found = (INPUT_DIR / d).exists()
    log(f"  {d}: {'✓' if found else '✗ MISSING!'}")

# Create symlink (instant, no copy)
RAW = Path("/kaggle/working/data/raw/ASVspoof2019_LA")
if RAW.exists():
    log("Symlink already exists, skipping")
else:
    RAW.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(INPUT_DIR), str(RAW), target_is_directory=True)
    log(f"Symlink created: {RAW} -> {INPUT_DIR}")

# Quick sanity check
flac_files = list(RAW.glob("ASVspoof2019_LA_train/flac/*.flac"))
log(f"Train FLAC files found: {len(flac_files)}")
assert len(flac_files) > 1000, f"Too few audio files ({len(flac_files)}), dataset may be incomplete"
log("Dataset ready ✓")

In [ ]:
# ── Install AuralGuard ──
log("Starting installation...")

# Clone repo (must be public for unauthenticated HTTPS)
REPO_DIR = Path("/kaggle/working/auralguard")
if REPO_DIR.exists():
    log("Repo already cloned, pulling latest...")
    run_cmd("git pull", cwd=str(REPO_DIR))
else:
    log("Cloning repo (public)...")
    ok = run_cmd(
        "git clone https://github.com/MIHMahmudEli/auralguard.git /kaggle/working/auralguard",
        timeout_min=5
    )
    if not ok:
        log("Clone failed -- check repo visibility or network")
        raise RuntimeError("Clone failed")

os.chdir(str(REPO_DIR))
log(f"Working dir: {os.getcwd()}")

# Install dependencies
log("Installing AuralGuard + pip deps...")
run_cmd("pip install -e .[train,dev]", timeout_min=10)
run_cmd("pip install datasets", timeout_min=5)

import torch
log(f"PyTorch {torch.__version__} -- CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    log(f"  GPU: {torch.cuda.get_device_name(0)}  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
log("Installation complete ✓")

In [ ]:
# ── Build manifests ──
import pandas as pd

RAW = Path("/kaggle/working/data/raw/ASVspoof2019_LA")
MANIFEST_DIR = Path("/kaggle/working/data/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = ["utt_id", "path", "label", "attack", "dataset", "lang", "split", "codec"]

for split, suffix in [("train", "trn"), ("dev", "trl"), ("eval", "trl")]:
    proto = RAW / "ASVspoof2019_LA_cm_protocols" / f"ASVspoof2019.LA.cm.{split}.{suffix}.txt"
    audio_dir = RAW / f"ASVspoof2019_LA_{split}" / "flac"
    log(f"Building {split} manifest...")
    log(f"  Protocol: {proto}")
    log(f"  Audio:    {audio_dir}")

    assert proto.exists(), f"Protocol not found: {proto}"
    assert audio_dir.exists(), f"Audio dir not found: {audio_dir}"

    lines = proto.read_text().splitlines()
    rows = []
    for line in tqdm(lines, desc=f"{split}", unit="utt"):
        parts = line.split()
        utt, attack, key = parts[1], parts[3], parts[4]
        rows.append({
            "utt_id": utt,
            "path": str(audio_dir / f"{utt}.flac"),
            "label": 0 if key == "bonafide" else 1,
            "attack": "bonafide" if key == "bonafide" else attack,
            "dataset": "asvspoof2019_la",
            "lang": "en",
            "split": split,
            "codec": "none",
        })
    df = pd.DataFrame(rows, columns=COLUMNS)
    out = MANIFEST_DIR / f"asvspoof2019_la_{split}.csv"
    df.to_csv(out, index=False)
    log(f"  -> {out.name}: {len(df)} rows ({df.label.sum()} spoof, {len(df) - df.label.sum()} bona-fide)")

log("All manifests built ✓")

In [ ]:
# ── Build augmentation manifests (RIRs) ──
MANIFEST_DIR = Path("/kaggle/working/data/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

rir_root = Path("/kaggle/working/data/raw/RIRS_NOISES")
if not rir_root.exists():
    log("RIRS_NOISES not at working path, checking dataset root...")
    for p in Path("/kaggle/input").rglob("*RIR*"):
        if p.is_dir():
            rir_root = p
            log(f"  Found RIRs at {rir_root}")
            break

if rir_root.exists():
    rows = []
    rir_dirs = [
        "real_rirs_isotropic_noises/real_rirs_isotropic_noises",
        "simulated_rirs/mediumroom", "simulated_rirs/largeroom",
        "simulated_rirs/smallroom",
    ]
    for rir_dir in rir_dirs:
        d = rir_root / rir_dir
        if d.exists():
            files = list(d.rglob("*.wav"))
            rows.extend({"path": str(f)} for f in files)
            log(f"  {rir_dir}: {len(files)} files")
        else:
            log(f"  {rir_dir}: not found, skipping")
    pd.DataFrame(rows).to_csv(MANIFEST_DIR / "rirs.csv", index=False)
    log(f"RIRs manifest: {len(rows)} files ✓")
else:
    log("RIRs not found anywhere -- reverb augmentation disabled")

pd.DataFrame(columns=["path"]).to_csv(MANIFEST_DIR / "musan.csv", index=False)
log("MUSAN manifest: empty (not available on Kaggle)")
log("Augmentation manifests ready ✓")

## 2. Train Baselines
Each cell trains one model. **Run sequentially.**
If a cell runs 2x longer than estimated, abort and debug.

**Expected times:** B1=30min, B2=1hr, B3=2hr, B5=4hr, AuralGuard=6hr

In [ ]:
# ===========================================================
os.chdir("/kaggle/working/auralguard")
log("============================================================")
log("B1 (LFCC-LCNN) starting -- estimated 1 hrs")
log("============================================================")
run_cmd(
    "python scripts/train.py experiment=b1_lcnn data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv",
    timeout_min=60
)
log("B1_LCNN DONE ✓")


In [ ]:
# ===========================================================
os.chdir("/kaggle/working/auralguard")
log("============================================================")
log("B2 (RawNet2) starting -- estimated 2 hrs")
log("============================================================")
run_cmd(
    "python scripts/train.py experiment=b2_rawnet2 data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv",
    timeout_min=120
)
log("B2_RAWNET2 DONE ✓")


In [ ]:
# ===========================================================
os.chdir("/kaggle/working/auralguard")
log("============================================================")
log("B3 (AASIST) starting -- estimated 3 hrs")
log("============================================================")
run_cmd(
    "python scripts/train.py experiment=b3_aasist data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv",
    timeout_min=180
)
log("B3_AASIST DONE ✓")


In [ ]:
# ===========================================================
os.chdir("/kaggle/working/auralguard")
log("============================================================")
log("B5 (WavLM+OCS) starting -- estimated 6 hrs")
log("============================================================")
run_cmd(
    "python scripts/train.py experiment=b5_wavlm_ocs data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv",
    timeout_min=360
)
log("B5_WAVLM_OCS DONE ✓")


In [ ]:
# ===========================================================
os.chdir("/kaggle/working/auralguard")
log("============================================================")
log("AuralGuard (full) starting -- estimated 8 hrs")
log("============================================================")
run_cmd(
    "python scripts/train.py experiment=auralguard data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv",
    timeout_min=480
)
log("AURALGUARD DONE ✓")


## 3. Evaluate & Download Results
After training, evaluate all checkpoints.

In [ ]:
# ── Evaluate all experiments (in-domain) ──
os.chdir("/kaggle/working/auralguard")
log("Evaluating all trained models on in-domain test set...")

model_dirs = sorted([p for p in Path("experiments").iterdir() if p.is_dir()])
log(f"Found {len(model_dirs)} experiment(s): {[p.name for p in model_dirs]}")

for exp_dir in tqdm(model_dirs, desc="In-domain eval"):
    ckpt = exp_dir / "checkpoints" / "best.ckpt"
    if not ckpt.exists():
        log(f"  [skip] {exp_dir.name} -- no best.ckpt")
        continue
    out_dir = exp_dir / "eval_results"
    log(f"  Evaluating {exp_dir.name}...")
    run_cmd([
        "python", "scripts/evaluate.py",
        f"--ckpt={ckpt}",
        f"--out={out_dir}",
    ], timeout_min=30)

log("All in-domain evaluations complete ✓")

In [ ]:
# ── Eval all zero-shot ──
os.chdir("/kaggle/working/auralguard")
log("Evaluating all trained models on zero-shot corpora...")

model_dirs = sorted([p for p in Path("experiments").iterdir() if p.is_dir()])

for exp_dir in tqdm(model_dirs, desc="Zero-shot eval"):
    ckpt = exp_dir / "checkpoints" / "best.ckpt"
    if not ckpt.exists():
        log(f"  [skip] {exp_dir.name} -- no best.ckpt")
        continue
    out_dir = exp_dir / "zeroshot_results"
    log(f"  Zero-shot eval for {exp_dir.name}...")
    run_cmd([
        "python", "scripts/eval_all_zeroshot.py",
        f"--ckpt={ckpt}",
        f"--out={out_dir}",
    ], timeout_min=60)

log("All zero-shot evaluations complete ✓")

## 4. Paper Figures
Generate figures for the manuscript.

In [ ]:
# ── Generate paper figures ──
os.chdir("/kaggle/working/auralguard")

import json, math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"figure.dpi": 150, "font.size": 11})
FIGS = Path("/kaggle/working/paper_figures")
FIGS.mkdir(parents=True, exist_ok=True)

model_order = ["b1_lcnn", "b2_rawnet2", "b3_aasist", "b5_wavlm_ocs", "auralguard"]
display_names = {
    "b1_lcnn": "B1 (LFCC-LCNN)",
    "b2_rawnet2": "B2 (RawNet2)",
    "b3_aasist": "B3 (AASIST)",
    "b5_wavlm_ocs": "B5 (WavLM+OCS)",
    "auralguard": "AuralGuard",
}
colors = plt.cm.tab10(np.linspace(0, 1, len(model_order)))

# Training curves
log("[1/4] Training curves...")
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for idx, name in enumerate(tqdm(model_order, desc="Curves")):
        log_dir = Path("experiments") / name / "logs"
        if not log_dir.exists():
            continue
        try:
            ea = EventAccumulator(str(log_dir))
            ea.Reload()
            tags = ea.Tags().get("scalars", [])
            for ax, tag, ylabel in zip(axes, ["train/loss", "eval/eer"], ["Loss", "EER (%)"]):
                if tag in tags:
                    events = ea.Scalars(tag)
                    steps = [e.step for e in events]
                    vals = [e.value for e in events]
                    ax.plot(steps, vals, label=display_names.get(name, name),
                            color=colors[idx], linewidth=1.5)
                    ax.set_xlabel("Step")
                    ax.set_ylabel(ylabel)
                    ax.legend(fontsize=8, loc="upper right")
                    ax.grid(True, alpha=0.3)
        except Exception as e:
            log(f"  [skip] {name}: {e}")
    plt.tight_layout()
    fig.savefig(FIGS / "training_curves.png", bbox_inches="tight")
    plt.close()
    log(f"  -> {FIGS / 'training_curves.png'}")
except ImportError:
    log("  [skip] tensorboard not installed")

# EER bar chart
log("[2/4] EER bar chart...")
try:
    rows = []
    for name in model_order:
        res_file = Path("experiments") / name / "eval_results" / "results.json"
        if not res_file.exists():
            res_file = Path("experiments") / name / "zeroshot_results" / "results.json"
        if res_file.exists():
            data = json.loads(res_file.read_text())
            for ds_name, metrics in data.items():
                rows.append({"model": name, "dataset": ds_name, "eer": metrics["eer"]})
    if rows:
        df = pd.DataFrame(rows)
        dataset_order = ["in_domain_eval", "in_the_wild", "wavefake", "mlaad",
                        "asvspoof2021_la", "asvspoof2021_df"]
        df = df[df["dataset"].isin(dataset_order)]
        df["dataset"] = pd.Categorical(df["dataset"], categories=dataset_order, ordered=True)
        df["model"] = pd.Categorical(df["model"], categories=model_order, ordered=True)
        df = df.sort_values(["dataset", "model"])
        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(dataset_order))
        n_models = len(model_order)
        bar_width = 0.15
        for i, name in enumerate(model_order):
            subset = df[df["model"] == name]
            vals = [subset[subset["dataset"] == d]["eer"].values[0]
                    if len(subset[subset["dataset"] == d]) > 0 else 0 for d in dataset_order]
            offset = (i - n_models / 2 + 0.5) * bar_width
            ax.bar(x + offset, [v * 100 for v in vals], bar_width,
                   label=display_names.get(name, name), color=colors[i])
        ax.set_xticks(x)
        ax.set_xticklabels([d.replace("in_domain_eval", "In-Domain")
                             .replace("in_the_wild", "In-the-Wild")
                             .replace("wavefake", "WaveFake")
                             .replace("mlaad", "MLAAD")
                             .replace("asvspoof2021_la", "ASVspoof21 LA")
                             .replace("asvspoof2021_df", "ASVspoof21 DF")])
        ax.set_ylabel("EER (%)")
        ax.set_title("In-Domain & Zero-Shot EER Comparison")
        ax.legend(fontsize=8, loc="upper left")
        ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        fig.savefig(FIGS / "eer_comparison.png", bbox_inches="tight")
        plt.close()
        log(f"  -> {FIGS / 'eer_comparison.png'}")
except Exception as e:
    log(f"  [skip] EER bar chart: {e}")

# DET curves
log("[3/4] DET curves...")
try:
    from sklearn.metrics import roc_curve
    fig, ax = plt.subplots(figsize=(6, 6))
    for name, c in zip(model_order, colors):
        res_file = Path("experiments") / name / "eval_results" / "results.json"
        if not res_file.exists():
            continue
        data = json.loads(res_file.read_text())
        in_domain = data.get("in_domain_eval", {})
        if "scores" not in in_domain or "labels" not in in_domain:
            continue
        scores = np.array(in_domain["scores"])
        labels = np.array(in_domain["labels"])
        fpr, fnr, _ = roc_curve(labels, scores, pos_label=1)
        far, frr = fpr, fnr
        mask = (far > 1e-5) & (frr > 1e-5)
        ax.loglog(far[mask], frr[mask], label=display_names.get(name, name),
                  color=c, linewidth=1.5)
    ax.set_xlabel("False Alarm Rate (FAR)")
    ax.set_ylabel("False Rejection Rate (FRR)")
    ax.set_title("DET Curve -- In-Domain Evaluation")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, which="both")
    plt.tight_layout()
    fig.savefig(FIGS / "det_curve.png", bbox_inches="tight")
    plt.close()
    log(f"  -> {FIGS / 'det_curve.png'}")
except Exception as e:
    log(f"  [skip] DET curve: {e}")

# Score distribution
log("[4/4] Score distribution...")
try:
    res_file = Path("experiments") / "auralguard" / "eval_results" / "results.json"
    if res_file.exists():
        data = json.loads(res_file.read_text())
        in_domain = data.get("in_domain_eval", {})
        if "scores" in in_domain and "labels" in in_domain:
            scores = np.array(in_domain["scores"])
            labels = np.array(in_domain["labels"])
            fig, ax = plt.subplots(figsize=(7, 4))
            bonafide_scores = scores[labels == 0]
            spoof_scores = scores[labels == 1]
            ax.hist(bonafide_scores, bins=80, alpha=0.6, label="Bona-fide",
                    color="green", density=True)
            ax.hist(spoof_scores, bins=80, alpha=0.6, label="Spoof",
                    color="red", density=True)
            ax.axvline(0, color="black", linestyle="--", alpha=0.5, label="Decision boundary")
            ax.set_xlabel("Spoof Score")
            ax.set_ylabel("Density")
            ax.set_title("AuralGuard -- Score Distribution (In-Domain Eval)")
            ax.legend()
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            fig.savefig(FIGS / "score_distribution.png", bbox_inches="tight")
            plt.close()
            log(f"  -> {FIGS / 'score_distribution.png'}")
except Exception as e:
    log(f"  [skip] Score distribution: {e}")

log("All figures generated ✓")

In [ ]:
# ── Export results as markdown table for paper ──
os.chdir("/kaggle/working/auralguard")
log("Exporting results table...")

rows = []
for name in model_order:
    for suffix, label in [("eval_results", "In-Domain"), ("zeroshot_results", "Zero-Shot")]:
        res_file = Path("experiments") / name / suffix / "results.json"
        if res_file.exists():
            data = json.loads(res_file.read_text())
            for ds_name, m in data.items():
                rows.append({
                    "Model": display_names.get(name, name),
                    "Dataset": ds_name,
                    "EER": f"{m['eer']:.4f}",
                    "EER CI95": f"[{m.get('eer_ci95', [0,0])[0]:.4f}, {m.get('eer_ci95', [0,0])[1]:.4f}]",
                    "AUROC": f"{m['auroc']:.4f}",
                    "min t-DCF": f"{m['min_tdcf']:.4f}",
                    "F1": f"{m.get('f1', 0):.4f}",
                    "Bal. Acc": f"{m.get('balanced_accuracy', 0):.4f}",
                })
            log(f"  {name}/{suffix}: {len(data)} dataset(s)")

if rows:
    df = pd.DataFrame(rows)
    md = df.to_markdown(index=False)
    (FIGS / "results_table.md").write_text(md)
    df.to_csv(FIGS / "results_table.csv", index=False)
    log(f"Results table: {len(rows)} rows -> paper_figures/")
    print("\n" + md + "\n")
else:
    log("No results found to tabulate")

In [ ]:
# ── Package results & figures for download ──
import tarfile
from pathlib import Path

tar_path = "/kaggle/working/auralguard_results.tar.gz"
log(f"Packaging results -> {tar_path}...")

paths_to_add = []
for ckpt in Path("experiments").rglob("best.ckpt"):
    paths_to_add.append((str(ckpt), str(ckpt.relative_to("/kaggle/working"))))
log(f"  Checkpoints: {len(paths_to_add)}")

for res in Path("experiments").rglob("results.json"):
    paths_to_add.append((str(res), str(res.relative_to("/kaggle/working"))))
    log(f"  Result files: {len([p for p in paths_to_add if 'results.json' in p[0]])}")

fig_files = list(FIGS.glob("*"))
log(f"  Figures: {len(fig_files)}")
for fig in fig_files:
    paths_to_add.append((str(fig), f"paper_figures/{fig.name}"))

log(f"  Total items: {len(paths_to_add)}")

with tarfile.open(tar_path, "w:gz") as tar:
    for src, arcname in tqdm(paths_to_add, desc="Tar"):
        tar.add(src, arcname=arcname)

size = Path(tar_path).stat().st_size
log(f"Package created: {tar_path}")
log(f"  Size: {size / 1e6:.1f} MB")
log("  Download via Kaggle sidebar -> Output -> auralguard_results.tar.gz")